## 基础用法

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import os

llm = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}} # Deepseek 在思考模式下不支持结构化输出
)

class Person(BaseModel):
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")


llm_with_structured_output = llm.with_structured_output(Person)

person = llm_with_structured_output.invoke("张三，25岁，是一个软件工程师")
print(f"姓名: {person.name}")
print(f"年龄: {person.age}")
print(f"职业: {person.occupation}")
print(type(person))


姓名: 张三
年龄: 25
职业: 软件工程师
<class '__main__.Person'>


In [7]:
class SentimentAnalysis(BaseModel):
    """情感分析结果"""
    sentiment: str = Field(description="情感倾向：positive/negative/neutral")
    confidence: float = Field(description="置信度，0-1之间")
    keywords: list[str] = Field(description="关键词列表")
llm = init_chat_model(
    model="deepseek-v4-flash",
    api_key = os.getenv("DEEPSEEK_API_KEY"),
    base_url = os.getenv("DEEPSEEK_BASE_URL"),
    extra_body = {"thinking": {"type": "disabled"}}
)
llm_with_structured_output = llm.with_structured_output(SentimentAnalysis)

text = "这个课程内容很实用，学到了很多知识，强烈推荐！"
result = llm_with_structured_output.invoke(
    f"分析以下文本的情感：\n{text}"
)
print(f"类型: {type(result)}")
print(f"情感: {result.sentiment}")
print(f"置信度: {result.confidence}")
print(f"关键词: {result.keywords}")

类型: <class '__main__.SentimentAnalysis'>
情感: positive
置信度: 0.95
关键词: ['实用', '学到了很多知识', '强烈推荐']


## 可选与默认值

有的平台的模型不支持默认值，比如CloseAI的模型

In [8]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import os
from typing import Optional
load_dotenv()


llm = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}} # Deepseek 在思考模式下不支持结构化输出
)

class Person(BaseModel):
    name: str = Field(description="姓名")
    age: Optional[int] = Field(description="年龄")
    occupation: str = Field(description="职业")


llm_with_structured_output = llm.with_structured_output(Person)

person = llm_with_structured_output.invoke("张三，是一个软件工程师")
print(f"姓名: {person.name}", f"年龄: {person.age}", f"职业: {person.occupation}")
print(type(person))


姓名: 张三 年龄: None 职业: 软件工程师
<class '__main__.Person'>


In [9]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import os
from typing import Optional
load_dotenv()


llm = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}} # Deepseek 在思考模式下不支持结构化输出
)

class Person(BaseModel):
    name: str = Field(description="姓名")
    age: int = Field(default=20, description="年龄")
    occupation: str = Field(description="职业")


llm_with_structured_output = llm.with_structured_output(Person)

person = llm_with_structured_output.invoke("张三，是一个软件工程师")
print(f"姓名: {person.name}", f"年龄: {person.age}", f"职业: {person.occupation}")
print(type(person))


姓名: 张三 年龄: 20 职业: 软件工程师
<class '__main__.Person'>


## 枚举类型

包括Enum和Literal

In [10]:
from enum import Enum
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv
from typing import Optional
load_dotenv()


llm = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}} # Deepseek 在思考模式下不支持结构化输出
)

# 定义你的优先级枚举类
class Priority(str, Enum):
    LOW = "低"
    MEDIUM = "中"
    HIGH = "高"

class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Priority = Field(description="紧急程度")

# 测试
structured_llm = llm.with_structured_output(CustomerInfo)

conversation = """
客服：您好，请问有什么可以帮助您？
客户：我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服：好的，我帮您查一下
"""

result = structured_llm.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")
print(result)

name='王小明' phone='138-1234-5678' email='null' issue='订单一直没发货' urgency=<Priority.HIGH: '高'>


In [11]:
from enum import Enum
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv
from typing import Optional, Literal
load_dotenv()


llm = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}} # Deepseek 在思考模式下不支持结构化输出
)



class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Literal["低", "中", "高"] = Field(description="紧急程度")

# 测试
structured_llm = llm.with_structured_output(CustomerInfo)

conversation = """
客服：您好，请问有什么可以帮助您？
客户：我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服：好的，我帮您查一下
"""

result = structured_llm.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")
print(result)

name='王小明' phone='138-1234-5678' email='null' issue='订单一直没发货' urgency='高'


## 列表与嵌套

### 列表对象


In [14]:
from typing import List
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv
from typing import Optional, Literal
load_dotenv()

model = init_chat_model(
    model = "deepseek-v4-flash",
    api_key = os.getenv("DEEPSEEK_API_KEY"),
    base_url = os.getenv("DEEPSEEK_BASE_URL"),
    extra_body = {
        "thinking": {"type": "disabled"
        }
    }
)

class Person(BaseModel):
    """人物信息"""
    name: str
    age: int
class PersonList(BaseModel):
    """人物列表信息"""
    people: List[Person]  # 多个 Person 对象

structured_llm = model.with_structured_output(PersonList)
result = structured_llm.invoke("张三 30岁，李四 25岁")
print(result)

people=[Person(name='张三', age=30), Person(name='李四', age=25)]


### 列表字段

In [15]:
class Review(BaseModel):
    """产品评论"""
    product: str
    rating: int = Field(description="评分 1-5")
    pros: List[str] = Field(description="优点列表")
    cons: List[str] = Field(description="缺点列表")

structured_llm = model.with_structured_output(Review)

review = structured_llm.invoke("""iPhone 17 很棒！摄像头强大，手感好。但是价格贵，没有充电器。4分。
""")
print(review)

product='iPhone 17' rating=4 pros=['摄像头强大', '手感好'] cons=['价格贵', '没有充电器']


### 嵌套字段

In [16]:
class Address(BaseModel):
    """地点描述"""
    city: str
    district: str
class Company(BaseModel):
    """公司信息"""
    name: str
    address: Address  # 嵌套模型
structured_llm = model.with_structured_output(Company)
res = structured_llm.invoke("""公司名称：字节跳动;公司地址：北京市海淀区
""")
print(res)

name='字节跳动' address=Address(city='北京市', district='海淀区')


### 嵌套列表字段

In [ ]:
from typing import List
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv
load_dotenv()

model = init_chat_model(
    model = "deepseek-v4-flash",
    api_key = os.getenv("DEEPSEEK_API_KEY"),
    base_url = os.getenv("DEEPSEEK_BASE_URL"),
    extra_body = {
        "thinking": {"type": "disabled"
        }
    }
)
# 1. 定义嵌套的 Pydantic 模型
class Actor(BaseModel):
    """演员信息"""
    name: str = Field(description="演员姓名")
    role: str = Field(description="饰演的角色")
class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")
    director: str = Field(description="导演")
    cast: List[Actor] = Field(description="演员列表")  # 定义列表字段
    rating: float = Field(description="评分")
# 2. 初始化模型并绑定输出结构
structured_model = model.with_structured_output(Movie)
# 3. 调用模型，直接获取 Movie 实例
response = structured_model.invoke("请介绍电影《盗梦空间》")
# 4. 访问嵌套数据
print(f"电影名: {response.title}")
print(f"上映年份: {response.year}")
print(f"导演: {response.director}")
print(f"演员列表: {response.cast}")
print(f"评分: {response.rating}")

电影名: 盗梦空间
上映年份: 2010
导演: 克里斯托弗·诺兰
演员列表: [Actor(name='莱昂纳多·迪卡普里奥', role='道姆·柯布'), Actor(name='约瑟夫·高登-莱维特', role='亚瑟'), Actor(name='艾伦·佩吉', role='阿丽雅德妮'), Actor(name='汤姆·哈迪', role='伊姆斯'), Actor(name='渡边谦', role='斋藤'), Actor(name='玛丽昂·歌迪亚', role='梅尔'), Actor(name='基里安·墨菲', role='罗伯特·费舍尔'), Actor(name='迈克尔·凯恩', role='迈尔斯教授')]
评分: 9.3


In [21]:
from pydantic import BaseModel, Field
from typing import List


class Aspect(BaseModel):
    """评论维度"""
    name: str = Field(description="维度名称，如：质量、价格、服务")
    score: int = Field(description="评分，1-5")
    comment: str = Field(description="具体评价")


class ProductReview(BaseModel):
    """产品评论分析"""
    overall_sentiment: str = Field(description="整体情感：positive/negative/neutral")
    overall_score: int = Field(description="综合评分，1-5")
    aspects: List[Aspect] = Field(description="各维度评价")
    summary: str = Field(description="一句话总结")


# 创建结构化模型
structured_model = model.with_structured_output(ProductReview)
# 测试
review_text = """
这款笔记本电脑性能非常强大，运行大型软件毫无压力。
屏幕色彩鲜艳，看视频很舒服。
不过价格有点贵，而且风扇噪音较大。
客服态度很好，物流也快。
总体来说还是值得购买的。
"""

result = structured_model.invoke(
    f"分析以下产品评论：\n{review_text}"
)

print(f"整体情感：{result.overall_sentiment}")
print(f"综合评分：{result.overall_score}/5")
print(f"\n各维度评价：")
for aspect in result.aspects:
    print(f" - {aspect.name}: {aspect.score}/5 - {aspect.comment}")
print(f"\n总结：{result.summary}")

整体情感：positive
综合评分：4/5

各维度评价：
 - 性能: 5/5 - 性能非常强大，运行大型软件毫无压力
 - 屏幕: 5/5 - 屏幕色彩鲜艳，看视频很舒服
 - 价格: 3/5 - 价格有点贵
 - 噪音: 2/5 - 风扇噪音较大
 - 服务: 5/5 - 客服态度很好
 - 物流: 5/5 - 物流很快

总结：性能强劲、屏幕出色，但价格偏高且风扇噪音较大，整体值得购买。


## 带限制条件

In [22]:

from typing import List
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv
load_dotenv()

llm_ds = init_chat_model(
    model = "deepseek-v4-pro",
    api_key = os.getenv("DEEPSEEK_API_KEY"),
    base_url = os.getenv("DEEPSEEK_BASE_URL"),
    extra_body = {
        "thinking": {"type": "disabled"
        }
    }
)

llm_gpt = init_chat_model(
    model = "gpt-5.4-mini",
    api_key = os.getenv("OPENROUTER_API_KEY"),
    base_url = os.getenv("OPENROUTER_BASE_URL"),
)

class Product(BaseModel):
    """产品信息（严格验证）"""
    name: str = Field(description="产品名称（字符串类型）", min_length=2)
    price: float = Field(description="价格，数字类型", gt=0)
    stock: int = Field(description="库存，整数类型", ge=0)


structured_llm_ds = llm_ds.with_structured_output(Product)
structured_llm_gpt = llm_gpt.with_structured_output(Product)

In [27]:
try:
    res_ds = structured_llm_ds.invoke("华为mate 80 promax 价格是-7999，当前库存-100")
    print(res_ds) #  deepseek 没有原始错误数据做修正
except ValueError as e:
    print(e)



2 validation errors for Product
price
  Input should be greater than 0 [type=greater_than, input_value=-7999, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than
stock
  Input should be greater than or equal to 0 [type=greater_than_equal, input_value=-100, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than_equal


In [ ]:
try:
    res_gpt = structured_llm_gpt.invoke("华为mate 80 promax 价格是-7999，当前库存-100")
except ValueError as e:
    print(e)
print(res_gpt) #  gpt把原始错误数据修正了


name='华为mate 80 promax' price=7999.0 stock=100


# 本质还是JSON解析

Pydantic结构化输出四步骤

## 第1步：定义结构
```python
from pydantic import BaseModel, Field

class BookInfo(BaseModel):
    title: str = Field(description="书名")
    author: str = Field(description="作者名字")
    tags: list[str] = Field(description="书籍的标签或分类")
```

## 第2步：LangChain转换
LangChain 内部调用 Pydantic 底层方法 `model_json_schema()`，将Python类自动转为标准**JSON Schema**。
JSON Schema是规范JSON文本，会明确：字段名、数据类型（string/array等）、字段说明description。

## 第3步：模型交互与强约束
LangChain 将 JSON Schema 封装进大模型API请求：
1. 现代写法 `.with_structured_output`：OpenAI、Anthropic、Gemini 等主流大模型支持函数/工具调用、JSON模式，LangChain把Schema作为Tools参数传入。
2. 大模型侧强约束：如 OpenAI 开启 `strict=True`，启用**基于语法的采样约束**，模型生成Token时严格遵循JSON Schema语法树，从底层锁定输出格式不会错乱。

## 第4步：自动解析与验证
大模型返回合规JSON字符串后，由 `PydanticStructuredOutputParser` 解析器处理：
1. **解析**：把JSON字符串转为Python字典；
2. **校验**：字典灌入Pydantic模型，自动校验字段必填项、数据类型；缺字段/类型错误会抛出校验异常，可触发LangChain自动重试；
3. **结果返回**：校验通过后直接得到Pydantic实例对象，可直接用 `result.title` 点取属性，无需手动解析JSON。


In [3]:
### Pydantic 三模型结构化输出兼容性验证

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from pydantic import BaseModel, Field, ValidationError
from rich import print as rich_print
from typing import List
import os

load_dotenv(override=True)


class Actor(BaseModel):
    """演员信息"""
    name: str = Field(description="演员姓名")
    role: str = Field(description="饰演的角色")


class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")
    director: str = Field(description="导演")
    cast: List[Actor] = Field(description="演员列表")
    rating: float = Field(description="评分，满分十分")


def validate_movie_result(result) -> tuple[dict | None, list[str]]:
    """Pydantic 成功时应返回 Movie 实例；这里也兼容返回 dict 后再用 Movie 做二次校验。"""
    if isinstance(result, Movie):
        return result.model_dump(), []

    if isinstance(result, dict):
        try:
            return Movie.model_validate(result).model_dump(), []
        except ValidationError as e:
            return None, [str(e)]

    return None, [f"返回值既不是 Movie 实例，也不是 dict，而是 {type(result).__name__}"]


models = {
    "deepseek-v4-flash": init_chat_model(
        model="deepseek-v4-flash",
        extra_body={"thinking": {"type": "disabled"}},
    ),
    "deepseek-v4-pro": init_chat_model(
        model="deepseek-v4-pro",
        extra_body={"thinking": {"type": "disabled"}},
    ),
    "gpt-5.4-mini": init_chat_model(
        model="gpt-5.4-mini",
        model_provider="openai",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        base_url=os.getenv("OPENROUTER_BASE_URL"),
    ),
}


# 使用较弱提示词，不主动列出字段，用来测试模型/适配器是否真的会按 schema 补齐必填字段。
prompt = "请介绍电影《盗梦空间》"

for model_name, llm in models.items():
    print(f"\n===== {model_name} =====")
    try:
        structured_model = llm.with_structured_output(Movie)
        response = structured_model.invoke(prompt)
        data, errors = validate_movie_result(response)

        if errors:
            print("[FAIL] Pydantic 结构校验失败")
            rich_print(errors)
        else:
            print("[PASS] Pydantic 结构校验通过")
            rich_print(data)

        rich_print(response)

    except Exception as e:
        print("[FAIL] 调用失败")
        print(type(e).__name__, e)



===== deepseek-v4-flash =====
[FAIL] 调用失败
ValidationError 4 validation errors for Movie
year
  Field required [type=missing, input_value={'title': '盗梦空间'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
director
  Field required [type=missing, input_value={'title': '盗梦空间'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
cast
  Field required [type=missing, input_value={'title': '盗梦空间'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
rating
  Field required [type=missing, input_value={'title': '盗梦空间'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing

===== deepseek-v4-pro =====
[PASS] Pydantic 结构校验通过


{
    'title': '盗梦空间',
    'year': 2010,
    'director': '克里斯托弗·诺兰',
    'cast': [
        {'name': '莱昂纳多·迪卡普里奥', 'role': '道姆·柯布'},
        {'name': '约瑟夫·高登-莱维特', 'role': '亚瑟'},
        {'name': '艾伦·佩吉', 'role': '阿丽亚德妮'},
        {'name': '汤姆·哈迪', 'role': '伊姆斯'},
        {'name': '渡边谦', 'role': '斋藤'},
        {'name': '玛丽昂·歌迪亚', 'role': '梅尔'},
        {'name': '基里安·墨菲', 'role': '罗伯特·费舍尔'},
        {'name': '迈克尔·凯恩', 'role': '迈尔斯'}
    ],
    'rating': 9.3
}

Movie(
    title='盗梦空间',
    year=2010,
    director='克里斯托弗·诺兰',
    cast=[
        Actor(name='莱昂纳多·迪卡普里奥', role='道姆·柯布'),
        Actor(name='约瑟夫·高登-莱维特', role='亚瑟'),
        Actor(name='艾伦·佩吉', role='阿丽亚德妮'),
        Actor(name='汤姆·哈迪', role='伊姆斯'),
        Actor(name='渡边谦', role='斋藤'),
        Actor(name='玛丽昂·歌迪亚', role='梅尔'),
        Actor(name='基里安·墨菲', role='罗伯特·费舍尔'),
        Actor(name='迈克尔·凯恩', role='迈尔斯')
    ],
    rating=9.3
)


===== gpt-5.4-mini =====
[PASS] Pydantic 结构校验通过


{
    'title': '盗梦空间',
    'year': 2010,
    'director': '克里斯托弗·诺兰',
    'cast': [
        {'name': '莱昂纳多·迪卡普里奥', 'role': '多姆·柯布'},
        {'name': '约瑟夫·高登-莱维特', 'role': '亚瑟'},
        {'name': '艾伦·佩姬', 'role': '阿丽雅德妮'},
        {'name': '汤姆·哈迪', 'role': '伊姆斯'},
        {'name': '渡边谦', 'role': '斋藤'}
    ],
    'rating': 8.8
}

Movie(
    title='盗梦空间',
    year=2010,
    director='克里斯托弗·诺兰',
    cast=[
        Actor(name='莱昂纳多·迪卡普里奥', role='多姆·柯布'),
        Actor(name='约瑟夫·高登-莱维特', role='亚瑟'),
        Actor(name='艾伦·佩姬', role='阿丽雅德妮'),
        Actor(name='汤姆·哈迪', role='伊姆斯'),
        Actor(name='渡边谦', role='斋藤')
    ],
    rating=8.8
)